# Tarang v9.4 — Three-Phase Follow-Up on Cleaned Data

## v9.3 recap

v9.3 cleaned S annotations (relabeling non-premature "S" beats to N) and retrained. Results:

- **S precision doubled**: 0.156 → 0.271 (on cleaned test)
- **S recall barely moved**: 0.154 → 0.157
- **Val/test gap is 4.3×**: val S recall = 0.674, test S recall = 0.157

Label noise was real and cleaning helped precision. But the model still can't generalize S detection from 4 val patients to 24 test patients. The bottleneck is now **generalization**, not label noise.

## v9.4 tests three hypotheses

| Phase | Hypothesis | Test | Cost |
|---|---|---|---|
| A | The 0.95 cleaning threshold was too aggressive — it removed beats that shared morphology with real S | Sweep thresholds 0.85, 0.95, 1.05 | 2 SV retrains |
| B | The 4-patient val set is too small to give stable thresholds | 3-fold CV on 18 train patients (6 per fold) | 0 retrains (diagnostic) |
| C | The 130-sample window clips P-waves needed for S generalization | Window 200 on cleaned data | 1 gate + 1 SV retrain + re-extract |

Total: ~4 retrains + 1 re-extract. ~30 minutes.


## Setup — load v9.3 models and data

In [1]:
import os, json, glob, warnings
from collections import Counter
from datetime import datetime

import wfdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.signal import resample_poly
import tensorflow as tf
from tensorflow.keras import regularizers
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.utils import class_weight

warnings.filterwarnings('ignore')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)

BASE = 'C:/MMD Public/Hackathons/Team Ocelleon/dataset'
def find_dir(base, *candidates):
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p): return p
    for c in candidates:
        for v in [c, c.lower(), c.upper(), c.title()]:
            p = os.path.join(base, v)
            if os.path.isdir(p): return p
    return os.path.join(base, candidates[0])

MITBIH_PATH = find_dir(BASE, 'mit-bih-arrhythmia-database-1.0.0', 'mitbih')
SVDB_PATH   = find_dir(BASE, 'mit-bih-supraventricular-arrhythmia-database-1.0.0', 'svdb')
OUTPUTS_V9  = 'outputs_v9'
OUTPUTS_V93 = 'outputs_v93'
OUTPUTS_V94 = 'outputs_v94'
os.makedirs(OUTPUTS_V94, exist_ok=True)

SOURCE_FS, TARGET_FS = 360, 250
WINDOW_LEN = 130
SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

BEAT_MAP = {
    'N':'N','L':'N','R':'N','e':'N','j':'N',
    'A':'S','a':'S','J':'S','S':'S',
    'V':'V','E':'V','F':'V',
    '/':'Q','f':'Q','Q':'Q',
}
CLASSES_TO_USE = ['N','S','V']
MITBIH_ALL_RECORDS = [
    100,101,102,103,104,105,106,107,108,109,
    111,112,113,114,115,116,117,118,119,121,
    122,123,124,200,201,202,203,205,207,208,
    209,210,212,213,214,215,217,219,220,221,
    222,223,228,230,231,232,233,234
]
MITBIH_TEST_RECORDS = [
    101,106,108,109,112,114,115,116,118,119,
    201,202,203,205,207,208,209,210,217,219,
    221,223,228,231,233,234
]
MITBIH_VAL_RECORDS = [105, 124, 214, 220]
MITBIH_TRAIN_RECORDS = [r for r in MITBIH_ALL_RECORDS
                        if r not in MITBIH_TEST_RECORDS and r not in MITBIH_VAL_RECORDS]
SVDB_RECORDS = [str(i) for i in range(800, 895)]

# Load v9 data checkpoint
cp = np.load(f'{OUTPUTS_V9}/data_checkpoint.npz', allow_pickle=True)
mb_beats, mb_rr = cp['mb_beats'], cp['mb_rr']
mb_labels, mb_recs = cp['mb_labels'], cp['mb_recs']
sv_beats, sv_rr = cp['sv_beats'], cp['sv_rr']
sv_labels, sv_recs = cp['sv_labels'], cp['sv_recs']

X_all     = np.concatenate([mb_beats, sv_beats], axis=0)
X_rr_all  = np.concatenate([mb_rr,    sv_rr],   axis=0)
y_raw_all = np.concatenate([mb_labels, sv_labels])
recs_all  = np.concatenate([mb_recs,   sv_recs])
le = LabelEncoder(); le.fit(CLASSES_TO_USE)
y_all = le.transform(y_raw_all)
n_idx = int(np.where(le.classes_=='N')[0][0])
s_idx = int(np.where(le.classes_=='S')[0][0])
v_idx = int(np.where(le.classes_=='V')[0][0])

test_recs_set = {f'mitbih_{r}' for r in MITBIH_TEST_RECORDS}
val_recs_set  = {f'mitbih_{r}' for r in MITBIH_VAL_RECORDS}
test_mask  = np.isin(recs_all, list(test_recs_set))
val_mask   = np.isin(recs_all, list(val_recs_set))
train_mask = ~(test_mask | val_mask)

RR_MEAN = X_rr_all[train_mask].mean(axis=0)
RR_STD  = X_rr_all[train_mask].std(axis=0); RR_STD[RR_STD < 1e-8] = 1e-8
X_rr_all_norm = (X_rr_all - RR_MEAN) / RR_STD

# Load v9.3 gate (trained at 0.95 cleaning — we'll reuse it for Phase A)
gate_v93 = tf.keras.models.load_model(f'{OUTPUTS_V93}/gate_v93.keras', compile=False)
# Load v9.3 SV (the 0.95 baseline)
sv_v93_095 = tf.keras.models.load_model(f'{OUTPUTS_V93}/sv_v93.keras', compile=False)

print(f"Loaded v9.3 gate ({gate_v93.count_params():,} params)")
print(f"Loaded v9.3 SV baseline at 0.95 cleaning ({sv_v93_095.count_params():,} params)")

# v9.3 thresholds (from the joint sweep on 0.95-cleaned val)
V93_GATE_THR, V93_V_THR, V93_S_THR = 0.100, 0.20, 0.50

# Helper functions
def augment_batch(X_c, X_rr_c, class_name, n_copies, rng=None):
    if rng is None: rng = np.random.default_rng(SEED)
    n = len(X_c)
    if n == 0 or n_copies == 0:
        return (np.empty((0,)+X_c.shape[1:], dtype=np.float32),
                np.empty((0,)+X_rr_c.shape[1:], dtype=np.float32))
    out_X = np.repeat(X_c, n_copies, axis=0)
    out_rr = np.repeat(X_rr_c, n_copies, axis=0)
    shift = rng.integers(-3, 4, size=len(out_X))
    out_X_aug = np.empty_like(out_X)
    for i, s in enumerate(shift):
        if s > 0: out_X_aug[i, :-s] = out_X[i, s:]; out_X_aug[i, -s:] = out_X[i, -1:]
        elif s < 0: out_X_aug[i, -s:] = out_X[i, :s]; out_X_aug[i, :-s] = out_X[i, :1]
        else: out_X_aug[i] = out_X[i]
    amp = rng.uniform(0.85, 1.15, size=(len(out_X), 1, 1)).astype(np.float32)
    out_X_aug *= amp
    out_X_aug += rng.normal(0, 0.02, size=out_X_aug.shape).astype(np.float32)
    if class_name == 'S':
        out_rr[:, 0] *= rng.uniform(0.55, 0.85, size=len(out_rr))
        out_rr[:, 1] *= rng.uniform(1.10, 1.40, size=len(out_rr))
        out_rr[:, 4] *= rng.uniform(1.20, 2.00, size=len(out_rr))
        out_rr[:, 6] = out_rr[:, 0] / np.maximum(out_rr[:, 3], 1e-4)
        out_rr[:, 5] = out_rr[:, 1] / np.maximum(out_rr[:, 3], 1e-4)
    elif class_name == 'V':
        out_rr[:, 1] *= rng.uniform(1.20, 1.60, size=len(out_rr))
        out_rr[:, 4] *= rng.uniform(1.10, 1.80, size=len(out_rr))
        out_rr[:, 6] = out_rr[:, 0] / np.maximum(out_rr[:, 3], 1e-4)
        out_rr[:, 5] = out_rr[:, 1] / np.maximum(out_rr[:, 3], 1e-4)
    return out_X_aug, out_rr

def build_sv_model(ecg_shape=(WINDOW_LEN,2), rr_shape=(7,)):
    ecg_in = tf.keras.Input(shape=ecg_shape, name='ecg_input')
    x = tf.keras.layers.Reshape((WINDOW_LEN,2,1))(ecg_in)
    for f,k,d in [(16,7,0.1),(32,5,0.1),(48,5,0.15),(48,3,0.0)]:
        x = tf.keras.layers.Conv2D(f,(k,1),padding='same',use_bias=False,
                kernel_regularizer=regularizers.l2(1e-4))(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        if k >= 5: x = tf.keras.layers.MaxPooling2D((2,1))(x); x = tf.keras.layers.SpatialDropout2D(d)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    rr_in = tf.keras.Input(shape=rr_shape, name='rr_input')
    r = tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(rr_in)
    r = tf.keras.layers.Dropout(0.2)(r)
    r = tf.keras.layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(r)
    m = tf.keras.layers.Concatenate()([x, r])
    m = tf.keras.layers.Dense(32, use_bias=False)(m)
    m = tf.keras.layers.BatchNormalization()(m)
    m = tf.keras.layers.Activation('relu')(m)
    m = tf.keras.layers.Dropout(0.35)(m)
    v = tf.keras.layers.Dense(1, activation='sigmoid', name='v_head')(m)
    s = tf.keras.layers.Dense(1, activation='sigmoid', name='s_head')(m)
    return tf.keras.Model(inputs=[ecg_in,rr_in], outputs=[v,s])

def joint_threshold_sweep(gate_probs, v_probs, s_probs, y_true):
    GATE_THRS = np.arange(0.10, 0.71, 0.025)
    V_THRS    = np.arange(0.10, 0.95, 0.05)
    S_THRS    = np.arange(0.10, 0.95, 0.05)
    rows = []
    y_t = y_true.astype(np.int8)
    for g_thr in GATE_THRS:
        routed = gate_probs > g_thr
        if routed.sum() == 0: continue
        for v_thr in V_THRS:
            v_claim = routed & (v_probs > v_thr)
            for s_thr in S_THRS:
                s_claim = routed & (~v_claim) & (s_probs > s_thr)
                y_p = np.full_like(y_t, n_idx)
                y_p[v_claim] = v_idx; y_p[s_claim] = s_idx
                flat = (y_t * 3 + y_p).astype(np.int32)
                cm = np.bincount(flat, minlength=9).reshape(3, 3)
                f1s, recalls, precisions = [], [], []
                for i in range(3):
                    tp = int(cm[i,i]); fn = int(cm[i,:].sum()-tp); fp = int(cm[:,i].sum()-tp)
                    rec = tp/max(tp+fn,1); prec = tp/max(tp+fp,1)
                    f1s.append(2*prec*rec/max(prec+rec,1e-7))
                    recalls.append(rec); precisions.append(prec)
                rows.append({'gate_thr':float(g_thr),'v_thr':float(v_thr),'s_thr':float(s_thr),
                              'macro_f1':float(np.mean(f1s)),
                              's_recall':recalls[s_idx],'s_precision':precisions[s_idx],
                              'v_recall':recalls[v_idx],'n_precision':precisions[n_idx]})
    return pd.DataFrame(rows)

def eval_cascade(gate_model, sv_model, X, X_rr_norm, y_true, gate_thr, v_thr, s_thr, le, name=""):
    gate_probs = gate_model.predict([X, X_rr_norm], batch_size=256, verbose=0).flatten()
    gate_pass = gate_probs > gate_thr
    y_pred = np.full(len(y_true), n_idx, dtype=int)
    if gate_pass.any():
        v_p, s_p = sv_model.predict([X[gate_pass], X_rr_norm[gate_pass]], batch_size=256, verbose=0)
        v_p = v_p.flatten(); s_p = s_p.flatten()
        routed = np.full(len(v_p), n_idx, dtype=int)
        v_fire = v_p > v_thr; routed[v_fire] = v_idx
        s_fire = (~v_fire) & (s_p > s_thr); routed[s_fire] = s_idx
        y_pred[gate_pass] = routed
    cm = confusion_matrix(y_true, y_pred, labels=[n_idx, s_idx, v_idx])
    m = {'set': name, 'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0))}
    for i, cls in enumerate(le.classes_):
        tp = int(cm[i,i]); fn = int(cm[i,:].sum()-tp); fp = int(cm[:,i].sum()-tp)
        total = int(cm[i,:].sum())
        rec = tp/max(total,1); prec = tp/max(tp+fp,1)
        m[f'{cls}_f1'] = 2*prec*rec/max(prec+rec,1e-7)
        m[f'{cls}_recall'] = rec; m[f'{cls}_precision'] = prec; m[f'{cls}_total'] = total
    return m

ALL_RESULTS = []

print("Setup complete.")


Loaded v9.3 gate (28,633 params)
Loaded v9.3 SV baseline at 0.95 cleaning (20,090 params)
Setup complete.


## Phase A — Cleaning threshold sweep (0.85 / 0.95 / 1.05)

v9.3 used threshold 0.95: any "S" beat with `prematurity_index >= 0.95` was relabeled to N. This removed 41% of test S beats and 24% of train S beats. The per-patient breakdown showed v9.3 **lost** mitbih_118 and mitbih_234 entirely — patients where v9 was catching S beats. The 0.95 threshold may have removed beats that shared morphology with real S.

**Three thresholds:**
- **0.85** (very strict): keep only genuinely premature S. Removes 53% of test S.
- **0.95** (v9.3 baseline): keep genuinely + mildly premature. Removes 41%.
- **1.05** (lenient): keep all except clear escape rhythms. Removes only 5.7%.

**Caveat:** for efficiency, all three SV heads use the v9.3 gate (trained at 0.95 cleaning). The gate's N-vs-SV distinction changes minimally with cleaning threshold (<1% of beats shift), so this is a reasonable approximation. The SV head is where the S/N distinction matters.

For each threshold: clean labels → build SV training set → retrain SV → joint sweep → eval on cleaned test.

In [2]:
def clean_labels(prematurity_threshold):
    """Clean S annotations: relabel S beats with prematurity_index >= threshold to N."""
    premat = X_rr_all[:, 6]
    relabel = (y_raw_all == 'S') & (premat >= prematurity_threshold)
    y_raw_clean = y_raw_all.copy()
    y_raw_clean[relabel] = 'N'
    return le.transform(y_raw_clean), y_raw_clean

def train_sv_for_threshold(threshold, y_clean, y_raw_clean):
    """Train SV head for a given cleaning threshold. Uses v9.3 gate."""
    print(f"\n--- Training SV for cleaning threshold {threshold} ---")
    
    # Splits
    y_train = y_clean[train_mask]; y_train_raw = y_raw_clean[train_mask]
    y_val = y_clean[val_mask]; y_val_raw = y_raw_clean[val_mask]
    y_test = y_clean[test_mask]
    
    X_train = X_all[train_mask]; X_rr_train = X_rr_all_norm[train_mask]
    X_val = X_all[val_mask]; X_rr_val = X_rr_all_norm[val_mask]
    X_test = X_all[test_mask]; X_rr_test = X_rr_all_norm[test_mask]
    
    # Build SV training set using v9.3 gate
    gate_probs_train = gate_v93.predict([X_train, X_rr_train], batch_size=256, verbose=0).flatten()
    routed_mask = gate_probs_train > V93_GATE_THR
    sv_X = X_train[routed_mask]; sv_rr = X_rr_train[routed_mask]
    sv_y = y_train[routed_mask]; sv_y_raw = y_train_raw[routed_mask]
    
    n_per = Counter(sv_y)
    n_n = n_per.get(n_idx,0); n_s = n_per.get(s_idx,0); n_v = n_per.get(v_idx,0)
    target_sv = max(n_s, n_v)
    target_n = int(round((0.50 / 0.50) * target_sv))
    print(f"  Routed: N={n_n}, S={n_s}, V={n_v}  targets: sv={target_sv}, n={target_n}")
    
    sv_X_list = [sv_X]; sv_rr_list = [sv_rr]; sv_y_list = [sv_y]
    for class_idx, class_name in enumerate(le.classes_):
        n_have = n_per.get(class_idx, 0)
        n_need_target = target_n if class_idx == n_idx else target_sv
        n_need = max(0, n_need_target - n_have)
        if n_need == 0 or n_have == 0: continue
        mask = sv_y == class_idx
        X_c, rr_c = sv_X[mask], sv_rr[mask]
        n_copies = max(1, n_need // n_have + (1 if n_need % n_have else 0))
        n_copies = min(n_copies, 10)
        X_aug, rr_aug = augment_batch(X_c, rr_c, class_name, n_copies)
        y_aug = np.full(len(X_aug), class_idx, dtype=sv_y.dtype)
        sv_X_list.append(X_aug); sv_rr_list.append(rr_aug); sv_y_list.append(y_aug)
    
    sv_X_aug = np.concatenate(sv_X_list); sv_rr_aug = np.concatenate(sv_rr_list)
    sv_y_aug = np.concatenate(sv_y_list)
    perm = np.random.permutation(len(sv_X_aug))
    sv_X_aug = sv_X_aug[perm]; sv_rr_aug = sv_rr_aug[perm]; sv_y_aug = sv_y_aug[perm]
    
    # Val (gate-routed)
    gate_probs_val = gate_v93.predict([X_val, X_rr_val], batch_size=256, verbose=0).flatten()
    routed_val = gate_probs_val > V93_GATE_THR
    sv_X_val = X_val[routed_val]; sv_rr_val = X_rr_val[routed_val]; sv_y_val = y_val[routed_val]
    
    # Train
    model = build_sv_model()
    y_v_aug = (sv_y_aug == v_idx).astype(np.float32)
    y_s_aug = (sv_y_aug == s_idx).astype(np.float32)
    y_v_val = (sv_y_val == v_idx).astype(np.float32)
    y_s_val = (sv_y_val == s_idx).astype(np.float32)
    cw_v = class_weight.compute_class_weight('balanced', classes=np.array([0,1]), y=y_v_aug.astype(int))
    cw_s = class_weight.compute_class_weight('balanced', classes=np.array([0,1]), y=y_s_aug.astype(int))
    sw_v = np.where(y_v_aug==1, cw_v[1], cw_v[0]).astype(np.float32)
    sw_s = np.where(y_s_aug==1, cw_s[1], cw_s[0]).astype(np.float32)
    
    class CB(tf.keras.callbacks.Callback):
        def __init__(self, vd, yv, ys): self.vd=vd; self.yv=yv; self.ys=ys
        def on_epoch_end(self, e, logs=None):
            logs = logs or {}
            v,s = self.model.predict(self.vd, verbose=0)
            v=v.flatten(); s=s.flatten()
            vp=(v>0.5).astype(int); sp=(s>0.5).astype(int)
            def pr(yt,yp):
                tp=np.sum((yt==1)&(yp==1));fp=np.sum((yt==0)&(yp==1));fn=np.sum((yt==1)&(yp==0))
                return tp/max(tp+fp,1),tp/max(tp+fn,1)
            vpr,vr=pr(self.yv,vp); spr,sr=pr(self.ys,sp)
            logs['val_combined_score']=float(0.5*(vr+sr))
    
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
        loss={'v_head':'binary_crossentropy','s_head':'binary_crossentropy'},
        metrics={'v_head':[tf.keras.metrics.AUC(name='auc')],'s_head':[tf.keras.metrics.AUC(name='auc')]})
    cbs = [
        CB([sv_X_val, sv_rr_val], y_v_val, y_s_val),
        tf.keras.callbacks.ModelCheckpoint(f'{OUTPUTS_V94}/sv_thr{threshold}.keras',
            monitor='val_combined_score', mode='max', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_combined_score', mode='max',
            patience=12, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_combined_score', mode='max',
            factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ]
    model.fit([sv_X_aug, sv_rr_aug], {'v_head':y_v_aug, 's_head':y_s_aug},
        sample_weight={'v_head':sw_v, 's_head':sw_s},
        validation_data=([sv_X_val, sv_rr_val], {'v_head':y_v_val, 's_head':y_s_val}),
        epochs=60, batch_size=256, callbacks=cbs, verbose=2)
    
    return model, y_test, X_test, X_rr_test

# ── Run Phase A ──────────────────────────────────────────────────────────────
print("=" * 80)
print("PHASE A: CLEANING THRESHOLD SWEEP")
print("=" * 80)

phase_A_results = {}
for threshold in [0.85, 0.95, 1.05]:
    y_clean, y_raw_clean = clean_labels(threshold)
    n_test_s = int(np.sum((y_raw_clean == 'S') & test_mask))
    n_train_s = int(np.sum((y_raw_clean == 'S') & train_mask))
    print(f"\nThreshold {threshold}: test S={n_test_s}, train S={n_train_s}")
    
    if threshold == 0.95:
        # Already have this model from v9.3
        model = sv_v93_095
        y_test_local = y_clean[test_mask]
        X_test_local = X_all[test_mask]; X_rr_test_local = X_rr_all_norm[test_mask]
    else:
        model, y_test_local, X_test_local, X_rr_test_local = train_sv_for_threshold(
            threshold, y_clean, y_raw_clean)
    
    # Joint sweep on val
    X_val_local = X_all[val_mask]; X_rr_val_local = X_rr_all_norm[val_mask]
    y_val_local = y_clean[val_mask]
    gate_probs_val = gate_v93.predict([X_val_local, X_rr_val_local], batch_size=256, verbose=0).flatten()
    v_pv, s_pv = model.predict([X_val_local, X_rr_val_local], batch_size=256, verbose=0)
    v_pv = v_pv.flatten(); s_pv = s_pv.flatten()
    df_sweep = joint_threshold_sweep(gate_probs_val, v_pv, s_pv, y_val_local)
    best = df_sweep.loc[df_sweep['macro_f1'].idxmax()]
    
    # Eval on cleaned test
    test_metrics = eval_cascade(gate_v93, model, X_test_local, X_rr_test_local, y_test_local,
                                  float(best['gate_thr']), float(best['v_thr']), float(best['s_thr']),
                                  le, f"thr={threshold}")
    
    phase_A_results[threshold] = {
        'model': model, 'thresholds': best.to_dict(),
        'val_macro_f1': float(best['macro_f1']),
        'val_s_recall': float(best['s_recall']),
        'test_metrics': test_metrics,
        'n_test_s': n_test_s, 'n_train_s': n_train_s,
    }
    
    print(f"  Val: macro_f1={best['macro_f1']:.4f}, S recall={best['s_recall']:.3f}, S prec={best['s_precision']:.3f}")
    print(f"  Test: macro_f1={test_metrics['macro_f1']:.4f}, S F1={test_metrics['S_f1']:.3f}, "
          f"S recall={test_metrics['S_recall']:.3f}, S prec={test_metrics['S_precision']:.3f}")
    
    ALL_RESULTS.append({'experiment': f'A_thr{threshold}', **test_metrics})

# 

PHASE A: CLEANING THRESHOLD SWEEP

Threshold 0.85: test S=451, train S=8473

--- Training SV for cleaning threshold 0.85 ---
  Routed: N=2858, S=8448, V=11550  targets: sv=11550, n=11550
Epoch 1/60

Epoch 1: val_combined_score improved from -inf to 0.50000, saving model to outputs_v94\sv_thr0.85.keras
167/167 - 6s - loss: 1.1766 - v_head_loss: 0.6339 - s_head_loss: 0.5372 - v_head_auc: 0.6438 - s_head_auc: 0.7183 - val_loss: 1.2214 - val_v_head_loss: 0.6924 - val_s_head_loss: 0.5263 - val_v_head_auc: 0.5177 - val_s_head_auc: 0.5009 - val_combined_score: 0.5000 - lr: 0.0010 - 6s/epoch - 39ms/step
Epoch 2/60

Epoch 2: val_combined_score did not improve from 0.50000
167/167 - 2s - loss: 0.8960 - v_head_loss: 0.4108 - s_head_loss: 0.4820 - v_head_auc: 0.9116 - s_head_auc: 0.7956 - val_loss: 1.1698 - val_v_head_loss: 0.6728 - val_s_head_loss: 0.4933 - val_v_head_auc: 0.6489 - val_s_head_auc: 0.6635 - val_combined_score: 0.2500 - lr: 0.0010 - 2s/epoch - 14ms/step
Epoch 3/60

Epoch 3: val_com

KeyError: 'n_test_S'

## Phase B — 3-fold CV stability on cleaned data

v9.3's val S recall was 0.674 but test S recall was 0.157 — a 4.3× gap. Two possible explanations:

1. **The 4-patient val set is unrepresentative** — if the 4 val patients happen to have "easy" S beats, the model looks good on val but fails on the harder test patients.
2. **The model genuinely overfits to train-patient morphology** — even with a representative val set, the model can't generalize to unseen patients.

**3-fold CV distinguishes these.** Split the 18 train records into 3 folds of 6 patients each. For each fold, run the joint sweep using that fold as val. If the 3 folds pick similar thresholds and give similar S recall, the 4-patient val was the problem (6 patients is enough). If they disagree, the model is unstable regardless of val size.

**Cost: 0 retrains.** Uses the Phase A best model, just evaluates on 3 different val folds.

In [7]:
print(f"\n{'='*80}")
print(f"PHASE A SUMMARY — CLEANING THRESHOLD SWEEP")
print(f"{'='*80}")
print(f"{'Threshold':<12} {'n_test_s':>9} {'val_s_rec':>10} {'test_S_F1':>10} {'test_S_rec':>11} {'test_S_prec':>12} {'test_macro':>11}")
print(f"{'-'*78}")
for thr in [0.85, 0.95, 1.05]:
    r = phase_A_results[thr]
    tm = r['test_metrics']
    print(f"  {thr:<12} {r['n_test_s']:>9} {r['val_s_recall']:>10.3f} {tm['S_f1']:>10.3f} "
          f"{tm['S_recall']:>11.3f} {tm['S_precision']:>12.3f} {tm['macro_f1']:>11.4f}")

# Pick best
best_thr = max(phase_A_results.keys(), key=lambda t: phase_A_results[t]['test_metrics']['S_f1'])
print(f"\n  Best threshold by test S F1: {best_thr}")
BEST_THRESHOLD = best_thr
print(f"  → Using {BEST_THRESHOLD} for Phases B and C")

print("=" * 80)
print(f"PHASE B: 3-FOLD CV STABILITY (using best threshold {BEST_THRESHOLD} model)")
print("=" * 80)

# Build 3-fold partition (same as v9)
y_clean_best, y_raw_clean_best = clean_labels(BEST_THRESHOLD)

# Sort train records by S count (desc), deal round-robin
rec_s_counts = {}
for r in MITBIH_TRAIN_RECORDS:
    rec_tag = f'mitbih_{r}'
    mask = (recs_all == rec_tag) & train_mask
    rec_s_counts[r] = int(np.sum((y_raw_clean_best == 'S') & mask))
sorted_recs = sorted(MITBIH_TRAIN_RECORDS, key=lambda r: -rec_s_counts[r])

folds = {0: [], 1: [], 2: []}
for i, r in enumerate(sorted_recs):
    folds[i % 3].append(r)
for k in folds:
    folds[k].sort()
    fold_s = sum(rec_s_counts[r] for r in folds[k])
    print(f"  Fold {k} ({len(folds[k])} records): {folds[k]} — {fold_s} cleaned S beats")

# Use the best Phase A model
best_model = phase_A_results[BEST_THRESHOLD]['model']

fold_results = {}
for k in range(3):
    fold_recs = {f'mitbih_{r}' for r in folds[k]}
    fold_mask = np.isin(recs_all, list(fold_recs)) & train_mask
    
    X_fold = X_all[fold_mask]; X_rr_fold = X_rr_all_norm[fold_mask]
    y_fold = y_clean_best[fold_mask]
    
    print(f"\n--- Fold {k} ({int(fold_mask.sum())} beats, {int(np.sum(y_fold==s_idx))} S) ---")
    
    gate_probs = gate_v93.predict([X_fold, X_rr_fold], batch_size=256, verbose=0).flatten()
    v_p, s_p = best_model.predict([X_fold, X_rr_fold], batch_size=256, verbose=0)
    v_p = v_p.flatten(); s_p = s_p.flatten()
    
    df_sweep = joint_threshold_sweep(gate_probs, v_p, s_p, y_fold)
    best = df_sweep.loc[df_sweep['macro_f1'].idxmax()]
    
    fold_results[k] = {
        'best_triple': best.to_dict(),
        'val_macro_f1': float(best['macro_f1']),
        'val_s_recall': float(best['s_recall']),
        'val_s_precision': float(best['s_precision']),
        'n_patients': len(folds[k]),
        'n_s': int(np.sum(y_fold == s_idx)),
    }
    print(f"  Best: gate={best['gate_thr']:.3f}, V={best['v_thr']:.2f}, S={best['s_thr']:.2f} "
          f"→ macro_f1={best['macro_f1']:.4f}, S recall={best['s_recall']:.3f}")

# Cross-fold stability
print(f"\n{'='*80}")
print(f"CROSS-FOLD STABILITY")
print(f"{'='*80}")
print(f"{'Fold':<6} {'gate_thr':<10} {'v_thr':<8} {'s_thr':<8} {'macro_f1':<10} {'S_recall':<10} {'n_S':<6}")
print(f"{'-'*60}")
gate_thrs = []; v_thrs = []; s_thrs = []
for k in range(3):
    r = fold_results[k]
    b = r['best_triple']
    print(f"{k:<6} {b['gate_thr']:<10.3f} {b['v_thr']:<8.2f} {b['s_thr']:<8.2f} "
          f"{r['val_macro_f1']:<10.4f} {r['val_s_recall']:<10.3f} {r['n_s']:<6}")
    gate_thrs.append(b['gate_thr']); v_thrs.append(b['v_thr']); s_thrs.append(b['s_thr'])

print(f"\n  Spread (max-min): gate={max(gate_thrs)-min(gate_thrs):.3f}, "
      f"V={max(v_thrs)-min(v_thrs):.2f}, S={max(s_thrs)-min(s_thrs):.2f}")

# Also compare to v9.3's 4-patient val
print(f"\n  v9.3 4-patient val: S recall = {phase_A_results[BEST_THRESHOLD]['val_s_recall']:.3f}")
print(f"  3-fold mean S recall: {np.mean([fold_results[k]['val_s_recall'] for k in range(3)]):.3f}")
print(f"  Test S recall: {phase_A_results[BEST_THRESHOLD]['test_metrics']['S_recall']:.3f}")

# Verdict
fold_spread = max(gate_thrs) - min(gate_thrs)
mean_fold_recall = np.mean([fold_results[k]['val_s_recall'] for k in range(3)])
test_recall = phase_A_results[BEST_THRESHOLD]['test_metrics']['S_recall']

print(f"\n{'='*80}")
print(f"PHASE B VERDICT")
print(f"{'='*80}")
if fold_spread > 0.20:
    print(f"  ⚠ Thresholds UNSTABLE across folds (gate spread {fold_spread:.3f})")
    print(f"    → The model's optimal operating point depends on which patients you validate on")
    print(f"    → This is a generalization problem, not a val-size problem")
else:
    print(f"  ✓ Thresholds STABLE across folds (gate spread {fold_spread:.3f})")
    print(f"    → The 4-patient val was the problem; 6 patients gives stable thresholds")

if mean_fold_recall > test_recall * 2:
    print(f"  ⚠ Val S recall ({mean_fold_recall:.3f}) is {mean_fold_recall/test_recall:.1f}× the test S recall ({test_recall:.3f})")
    print(f"    → The model overfits to train-patient morphology even on held-out folds")
    print(f"    → More patient diversity in training is needed, OR the architecture can't generalize")
else:
    print(f"  ✓ Val/test gap is small ({mean_fold_recall:.3f} vs {test_recall:.3f})")
    print(f"    → Generalization is OK; the gap was a val-size artifact")
print(f"{'='*80}")


PHASE A SUMMARY — CLEANING THRESHOLD SWEEP
Threshold     n_test_s  val_s_rec  test_S_F1  test_S_rec  test_S_prec  test_macro
------------------------------------------------------------------------------
  0.85               451      0.795      0.178       0.162        0.198      0.5518
  0.95               566      0.674      0.199       0.157        0.271      0.5592
  1.05               906      0.573      0.131       0.098        0.197      0.5358

  Best threshold by test S F1: 0.95
  → Using 0.95 for Phases B and C
PHASE B: 3-FOLD CV STABILITY (using best threshold 0.95 model)
  Fold 0 (6 records): [102, 111, 123, 213, 215, 232] — 736 cleaned S beats
  Fold 1 (6 records): [103, 104, 117, 200, 212, 222] — 209 cleaned S beats
  Fold 2 (6 records): [100, 107, 113, 121, 122, 230] — 40 cleaned S beats

--- Fold 0 (12137 beats, 736 S) ---
  Best: gate=0.150, V=0.60, S=0.30 → macro_f1=0.9750, S recall=0.961

--- Fold 1 (11612 beats, 209 S) ---
  Best: gate=0.150, V=0.40, S=0.50 → macro

## Phase C — Window 200 samples on cleaned data

v9.1 C1 showed gate S-recall improved from 0.42 to 0.58 at 200 samples (on noisy labels). But S F1 went *down* because S precision dropped. The hypothesis: on cleaned data, the SV head can finally use the extra context (longer compensatory pause, visible P-waves) to distinguish real S from N.

**This is the KB's stated priority** (Conclusion #2: "S-specific data and feature improvement, not merely another architecture change"). With clean labels, it finally gets a fair test.

Re-extract all beats at 200 samples, retrain gate + SV from scratch on cleaned data, evaluate.

In [8]:
print("=" * 80)
print(f"PHASE C: WINDOW 200 SAMPLES ON CLEANED DATA (threshold {BEST_THRESHOLD})")
print("=" * 80)

WINDOW_LEN_C = 200
WINDOW_PRE_C = 100; WINDOW_POST_C = 100

def rolling_window_normalize(signal, fs, window_seconds=30):
    ws = int(window_seconds * fs)
    s = pd.Series(signal.astype(np.float64))
    roll = s.rolling(window=ws, min_periods=1)
    mean = roll.mean(); std = roll.std(ddof=0).fillna(0).clip(lower=1e-8)
    return ((s - mean) / std).values.astype(np.float32)

def compute_rr_features(peaks_sec, beat_idx):
    n = len(peaks_sec); i = beat_idx
    prev_idx = max(0, i - 1); next_idx = min(n - 1, i + 1)
    rr_prev = peaks_sec[i] - peaks_sec[prev_idx]
    rr_next = peaks_sec[next_idx] - peaks_sec[i]
    rr_ratio = rr_prev / max(rr_next, 1e-4)
    lo, hi = max(0, i - 2), min(n - 1, i + 2)
    local_rrs = np.diff(peaks_sec[lo:hi+1]).astype(np.float32)
    if len(local_rrs) == 0: rr_mean_5, rr_std_5 = rr_prev, 0.0
    else: rr_mean_5 = float(np.mean(local_rrs)); rr_std_5 = float(np.std(local_rrs))
    return np.array([rr_prev, rr_next, rr_ratio, rr_mean_5, rr_std_5,
                      rr_next/max(rr_mean_5,1e-4), rr_prev/max(rr_mean_5,1e-4)], dtype=np.float32)

def load_records_200(db_path, record_list, source_tag, two_channel=True):
    all_beats, all_rr, all_labels, all_recs = [], [], [], []
    for rec_id in record_list:
        rec_str = str(rec_id)
        try:
            record = wfdb.rdrecord(f'{db_path}/{rec_str}')
            annotation = wfdb.rdann(f'{db_path}/{rec_str}', 'atr')
            n_channels = record.p_signal.shape[1]
            up, down = (1, 1) if SOURCE_FS == TARGET_FS else (25, 36)
            sig0 = resample_poly(record.p_signal[:, 0], up, down)
            sig1 = resample_poly(record.p_signal[:, 1], up, down) if (two_channel and n_channels > 1) else sig0
            ecg_ch0 = rolling_window_normalize(sig0, TARGET_FS)
            ecg_ch1 = rolling_window_normalize(sig1, TARGET_FS)
            peak_idx = annotation.sample
            if SOURCE_FS != TARGET_FS:
                peak_idx = np.round(peak_idx * TARGET_FS / SOURCE_FS).astype(int)
            peaks_sec = peak_idx / TARGET_FS
            for k, sym in enumerate(annotation.symbol):
                if sym not in BEAT_MAP: continue
                aami = BEAT_MAP[sym]
                if aami not in CLASSES_TO_USE: continue
                center = peak_idx[k]
                lo = center - WINDOW_PRE_C; hi = center + WINDOW_POST_C
                if lo < 0 or hi >= len(ecg_ch0): continue
                beat = np.stack([ecg_ch0[lo:hi], ecg_ch1[lo:hi]], axis=-1).astype(np.float32)
                rr = compute_rr_features(peaks_sec, k)
                all_beats.append(beat); all_rr.append(rr)
                all_labels.append(aami); all_recs.append(f'{source_tag}_{rec_str}')
        except Exception as e:
            print(f"  [skip] {source_tag}_{rec_str}: {e}")
    if not all_beats:
        return (np.empty((0, WINDOW_LEN_C, 2), dtype=np.float32),
                np.empty((0, 7), dtype=np.float32),
                np.array([], dtype=object), np.array([], dtype=object))
    return (np.stack(all_beats), np.stack(all_rr),
            np.array(all_labels, dtype=object), np.array(all_recs, dtype=object))

# Re-extract
print("Re-extracting at 200 samples...")
mb_b200, mb_r200, mb_l200, mb_rec200 = load_records_200(MITBIH_PATH, MITBIH_ALL_RECORDS, 'mitbih')
sv_b200, sv_r200, sv_l200, sv_rec200 = load_records_200(SVDB_PATH, SVDB_RECORDS, 'svdb')
keep = sv_l200 != 'N'
sv_b200, sv_r200, sv_l200, sv_rec200 = sv_b200[keep], sv_r200[keep], sv_l200[keep], sv_rec200[keep]

X_200 = np.concatenate([mb_b200, sv_b200])
X_rr_200 = np.concatenate([mb_r200, sv_r200])
y_raw_200 = np.concatenate([mb_l200, sv_l200])
recs_200 = np.concatenate([mb_rec200, sv_rec200])

# Clean at best threshold
premat_200 = X_rr_200[:, 6]
relabel_200 = (y_raw_200 == 'S') & (premat_200 >= BEST_THRESHOLD)
y_raw_200_clean = y_raw_200.copy(); y_raw_200_clean[relabel_200] = 'N'
y_200 = le.transform(y_raw_200_clean)

# Splits
test_m = np.isin(recs_200, list(test_recs_set))
val_m = np.isin(recs_200, list(val_recs_set))
train_m = ~(test_m | val_m)

X_tr200 = X_200[train_m]; X_rr_tr200 = X_rr_200[train_m]
y_tr200 = y_200[train_m]; y_raw_tr200 = y_raw_200_clean[train_m]
X_va200 = X_200[val_m]; X_rr_va200 = X_rr_200[val_m]; y_va200 = y_200[val_m]
X_te200 = X_200[test_m]; X_rr_te200 = X_rr_200[test_m]; y_te200 = y_200[test_m]

RR_MEAN_200 = X_rr_tr200.mean(axis=0)
RR_STD_200 = X_rr_tr200.std(axis=0); RR_STD_200[RR_STD_200 < 1e-8] = 1e-8
X_rr_tr200_n = (X_rr_tr200 - RR_MEAN_200) / RR_STD_200
X_rr_va200_n = (X_rr_va200 - RR_MEAN_200) / RR_STD_200
X_rr_te200_n = (X_rr_te200 - RR_MEAN_200) / RR_STD_200

print(f"  Train: {X_tr200.shape}  S: {int(np.sum(y_raw_tr200=='S'))}")
print(f"  Val  : {X_va200.shape}  S: {int(np.sum(y_va200==s_idx))}")
print(f"  Test : {X_te200.shape}  S: {int(np.sum(y_te200==s_idx))}")

# Build gate model for 200 samples
def build_gate_200():
    ecg_in = tf.keras.Input(shape=(WINDOW_LEN_C, 2), name='ecg_input')
    x = tf.keras.layers.Reshape((WINDOW_LEN_C, 2, 1))(ecg_in)
    for f,k,d in [(16,7,0.1),(32,5,0.1),(64,5,0.15),(64,3,0.0)]:
        x = tf.keras.layers.Conv2D(f,(k,1),padding='same',use_bias=False,
                kernel_regularizer=regularizers.l2(1e-4))(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        if k >= 5: x = tf.keras.layers.MaxPooling2D((2,1))(x); x = tf.keras.layers.SpatialDropout2D(d)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    rr_in = tf.keras.Input(shape=(7,), name='rr_input')
    r = tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(rr_in)
    r = tf.keras.layers.Dropout(0.2)(r)
    r = tf.keras.layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(r)
    m = tf.keras.layers.Concatenate()([x, r])
    m = tf.keras.layers.Dense(32, use_bias=False)(m)
    m = tf.keras.layers.BatchNormalization()(m)
    m = tf.keras.layers.Activation('relu')(m)
    m = tf.keras.layers.Dropout(0.35)(m)
    return tf.keras.Model(inputs=[ecg_in,rr_in],
                           outputs=tf.keras.layers.Dense(1,activation='sigmoid')(m))

def build_sv_200():
    ecg_in = tf.keras.Input(shape=(WINDOW_LEN_C, 2), name='ecg_input')
    x = tf.keras.layers.Reshape((WINDOW_LEN_C, 2, 1))(ecg_in)
    for f,k,d in [(16,7,0.1),(32,5,0.1),(48,5,0.15),(48,3,0.0)]:
        x = tf.keras.layers.Conv2D(f,(k,1),padding='same',use_bias=False,
                kernel_regularizer=regularizers.l2(1e-4))(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation('relu')(x)
        if k >= 5: x = tf.keras.layers.MaxPooling2D((2,1))(x); x = tf.keras.layers.SpatialDropout2D(d)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    rr_in = tf.keras.Input(shape=(7,), name='rr_input')
    r = tf.keras.layers.Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(rr_in)
    r = tf.keras.layers.Dropout(0.2)(r)
    r = tf.keras.layers.Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(r)
    m = tf.keras.layers.Concatenate()([x, r])
    m = tf.keras.layers.Dense(32, use_bias=False)(m)
    m = tf.keras.layers.BatchNormalization()(m)
    m = tf.keras.layers.Activation('relu')(m)
    m = tf.keras.layers.Dropout(0.35)(m)
    v = tf.keras.layers.Dense(1, activation='sigmoid', name='v_head')(m)
    s = tf.keras.layers.Dense(1, activation='sigmoid', name='s_head')(m)
    return tf.keras.Model(inputs=[ecg_in,rr_in], outputs=[v,s])

# Augment for gate
ecg_list = [X_tr200]; rr_list = [X_rr_tr200_n]; y_list = [y_tr200]
n_per = Counter(y_tr200); n_tgt = max(n_per.values()) // 2
for ci, cn in enumerate(le.classes_):
    n_have = n_per.get(ci, 0); n_need = max(0, n_tgt - n_have)
    if n_need == 0: continue
    mask = y_tr200 == ci
    X_aug, rr_aug = augment_batch(X_tr200[mask], X_rr_tr200_n[mask], cn, max(1, n_need // n_have))
    ecg_list.append(X_aug); rr_list.append(rr_aug); y_list.append(np.full(len(X_aug), ci, dtype=y_tr200.dtype))
X_tr200_aug = np.concatenate(ecg_list); X_rr_tr200_aug = np.concatenate(rr_list)
y_tr200_aug = np.concatenate(y_list)
perm = np.random.permutation(len(X_tr200_aug))
X_tr200_aug = X_tr200_aug[perm]; X_rr_tr200_aug = X_rr_tr200_aug[perm]; y_tr200_aug = y_tr200_aug[perm]
y_tr200_gate = (y_tr200_aug != n_idx).astype(np.float32)
y_va200_gate = (y_va200 != n_idx).astype(np.float32)

# Train gate @ 200
print("\nTraining gate @ 200...")
gate_200 = build_gate_200()
gate_200.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[tf.keras.metrics.AUC(name='auc')])
gate_cw = class_weight.compute_class_weight('balanced', classes=np.unique(y_tr200_gate), y=y_tr200_gate)
gate_200.fit([X_tr200_aug, X_rr_tr200_aug], y_tr200_gate,
    validation_data=([X_va200, X_rr_va200_n], y_va200_gate),
    epochs=60, batch_size=256,
    class_weight={int(k):float(v) for k,v in zip(np.unique(y_tr200_gate), gate_cw)},
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max',
        patience=12, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(f'{OUTPUTS_V94}/gate_200.keras',
            monitor='val_auc', mode='max', save_best_only=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_auc', mode='max',
            factor=0.5, patience=5, min_lr=1e-6, verbose=1)],
    verbose=2)

# Gate threshold (v8.7 rule)
gate_probs_va200 = gate_200.predict([X_va200, X_rr_va200_n], batch_size=256, verbose=0).flatten()
v_recall_needed = 0.85 / 0.95
best_thr = None; best_np = -1
for thr in np.arange(0.10, 0.71, 0.025):
    pass_m = gate_probs_va200 > thr
    v_rec = np.sum((y_va200 == v_idx) & pass_m) / max(int(np.sum(y_va200 == v_idx)), 1)
    n_rec = np.sum((y_va200 == n_idx) & pass_m) / max(int(np.sum(y_va200 == n_idx)), 1)
    if v_rec >= v_recall_needed and (1 - n_rec) > best_np:
        best_np = 1 - n_rec; best_thr = thr
GATE_THR_200 = best_thr if best_thr else 0.10
print(f"Gate threshold @ 200: {GATE_THR_200:.3f}")

# Build SV training set
gate_probs_tr200 = gate_200.predict([X_tr200, X_rr_tr200_n], batch_size=256, verbose=0).flatten()
routed = gate_probs_tr200 > GATE_THR_200
sv_X = X_tr200[routed]; sv_rr = X_rr_tr200_n[routed]
sv_y = y_tr200[routed]; sv_y_raw = y_raw_tr200[routed]

n_per = Counter(sv_y)
n_n = n_per.get(n_idx,0); n_s = n_per.get(s_idx,0); n_v = n_per.get(v_idx,0)
target_sv = max(n_s, n_v); target_n = int(round((0.50/0.50)*target_sv))
sv_X_list = [sv_X]; sv_rr_list = [sv_rr]; sv_y_list = [sv_y]
for ci, cn in enumerate(le.classes_):
    n_have = n_per.get(ci, 0); n_need = max(0, (target_n if ci==n_idx else target_sv) - n_have)
    if n_need == 0 or n_have == 0: continue
    mask = sv_y == ci
    n_copies = min(10, max(1, n_need // n_have + (1 if n_need % n_have else 0)))
    X_aug, rr_aug = augment_batch(sv_X[mask], sv_rr[mask], cn, n_copies)
    sv_X_list.append(X_aug); sv_rr_list.append(rr_aug)
    sv_y_list.append(np.full(len(X_aug), ci, dtype=sv_y.dtype))
sv_X_aug = np.concatenate(sv_X_list); sv_rr_aug = np.concatenate(sv_rr_list)
sv_y_aug = np.concatenate(sv_y_list)
perm = np.random.permutation(len(sv_X_aug))
sv_X_aug = sv_X_aug[perm]; sv_rr_aug = sv_rr_aug[perm]; sv_y_aug = sv_y_aug[perm]

# Val
routed_va = gate_probs_va200 > GATE_THR_200
sv_X_va = X_va200[routed_va]; sv_rr_va = X_rr_va200_n[routed_va]; sv_y_va = y_va200[routed_va]

# Train SV @ 200
print("\nTraining SV @ 200...")
sv_200 = build_sv_200()
y_v = (sv_y_aug == v_idx).astype(np.float32); y_s = (sv_y_aug == s_idx).astype(np.float32)
y_vv = (sv_y_va == v_idx).astype(np.float32); y_sv = (sv_y_va == s_idx).astype(np.float32)
cw_v = class_weight.compute_class_weight('balanced', classes=np.array([0,1]), y=y_v.astype(int))
cw_s = class_weight.compute_class_weight('balanced', classes=np.array([0,1]), y=y_s.astype(int))
sw_v = np.where(y_v==1, cw_v[1], cw_v[0]).astype(np.float32)
sw_s = np.where(y_s==1, cw_s[1], cw_s[0]).astype(np.float32)

class CB200(tf.keras.callbacks.Callback):
    def __init__(self, vd, yv, ys): self.vd=vd; self.yv=yv; self.ys=ys
    def on_epoch_end(self, e, logs=None):
        logs = logs or {}
        v,s = self.model.predict(self.vd, verbose=0)
        v=v.flatten(); s=s.flatten()
        def pr(yt,yp):
            tp=np.sum((yt==1)&(yp==1));fp=np.sum((yt==0)&(yp==1));fn=np.sum((yt==1)&(yp==0))
            return tp/max(tp+fp,1),tp/max(tp+fn,1)
        _,vr=pr(self.yv,(v>0.5).astype(int)); _,sr=pr(self.ys,(s>0.5).astype(int))
        logs['val_combined_score']=float(0.5*(vr+sr))

sv_200.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
    loss={'v_head':'binary_crossentropy','s_head':'binary_crossentropy'},
    metrics={'v_head':[tf.keras.metrics.AUC(name='auc')],'s_head':[tf.keras.metrics.AUC(name='auc')]})
sv_200.fit([sv_X_aug, sv_rr_aug], {'v_head':y_v, 's_head':y_s},
    sample_weight={'v_head':sw_v, 's_head':sw_s},
    validation_data=([sv_X_va, sv_rr_va], {'v_head':y_vv, 's_head':y_sv}),
    epochs=60, batch_size=256,
    callbacks=[CB200([sv_X_va, sv_rr_va], y_vv, y_sv),
        tf.keras.callbacks.ModelCheckpoint(f'{OUTPUTS_V94}/sv_200.keras',
            monitor='val_combined_score', mode='max', save_best_only=True, verbose=1),
        tf.keras.callbacks.EarlyStopping(monitor='val_combined_score', mode='max',
            patience=12, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_combined_score', mode='max',
            factor=0.5, patience=5, min_lr=1e-6, verbose=1)],
    verbose=2)

# Joint sweep + eval
gate_probs_va200_full = gate_200.predict([X_va200, X_rr_va200_n], batch_size=256, verbose=0).flatten()
v_pv, s_pv = sv_200.predict([X_va200, X_rr_va200_n], batch_size=256, verbose=0)
v_pv = v_pv.flatten(); s_pv = s_pv.flatten()
df_sweep = joint_threshold_sweep(gate_probs_va200_full, v_pv, s_pv, y_va200)
best = df_sweep.loc[df_sweep['macro_f1'].idxmax()]
print(f"\nBest @ 200: gate={best['gate_thr']:.3f}, V={best['v_thr']:.2f}, S={best['s_thr']:.2f} "
      f"→ val macro_f1={best['macro_f1']:.4f}, S recall={best['s_recall']:.3f}")

test_200 = eval_cascade(gate_200, sv_200, X_te200, X_rr_te200_n, y_te200,
                         float(best['gate_thr']), float(best['v_thr']), float(best['s_thr']),
                         le, f"window=200, thr={BEST_THRESHOLD}")

print(f"\nPHASE C RESULT (window 200, cleaned at {BEST_THRESHOLD}):")
for cls in le.classes_:
    print(f"  {cls}: F1={test_200[f'{cls}_f1']:.3f}  Se={test_200[f'{cls}_recall']:.3f}  P+={test_200[f'{cls}_precision']:.3f}")
print(f"  Macro F1: {test_200['macro_f1']:.4f}")

# Compare to best 130-sample
best_130 = phase_A_results[BEST_THRESHOLD]['test_metrics']
print(f"\n  vs 130-sample (thr={BEST_THRESHOLD}):")
print(f"    S F1: {best_130['S_f1']:.3f} → {test_200['S_f1']:.3f} (Δ{test_200['S_f1']-best_130['S_f1']:+.3f})")
print(f"    S recall: {best_130['S_recall']:.3f} → {test_200['S_recall']:.3f} (Δ{test_200['S_recall']-best_130['S_recall']:+.3f})")
print(f"    S precision: {best_130['S_precision']:.3f} → {test_200['S_precision']:.3f} (Δ{test_200['S_precision']-best_130['S_precision']:+.3f})")
print(f"    Macro F1: {best_130['macro_f1']:.4f} → {test_200['macro_f1']:.4f} (Δ{test_200['macro_f1']-best_130['macro_f1']:+.4f})")

ALL_RESULTS.append({'experiment': f'C_window200_thr{BEST_THRESHOLD}', **test_200})


PHASE C: WINDOW 200 SAMPLES ON CLEANED DATA (threshold 0.95)
Re-extracting at 200 samples...
  [skip] svdb_813: [Errno 2] No such file or directory: 'C:\\MMD Public\\Hackathons\\Team Ocelleon\\dataset\\mit-bih-supraventricular-arrhythmia-database-1.0.0\\813.hea'
  [skip] svdb_814: [Errno 2] No such file or directory: 'C:\\MMD Public\\Hackathons\\Team Ocelleon\\dataset\\mit-bih-supraventricular-arrhythmia-database-1.0.0\\814.hea'
  [skip] svdb_815: [Errno 2] No such file or directory: 'C:\\MMD Public\\Hackathons\\Team Ocelleon\\dataset\\mit-bih-supraventricular-arrhythmia-database-1.0.0\\815.hea'
  [skip] svdb_816: [Errno 2] No such file or directory: 'C:\\MMD Public\\Hackathons\\Team Ocelleon\\dataset\\mit-bih-supraventricular-arrhythmia-database-1.0.0\\816.hea'
  [skip] svdb_817: [Errno 2] No such file or directory: 'C:\\MMD Public\\Hackathons\\Team Ocelleon\\dataset\\mit-bih-supraventricular-arrhythmia-database-1.0.0\\817.hea'
  [skip] svdb_818: [Errno 2] No such file or directory: '

## Final summary — all v9.4 experiments plus baselines

In [10]:
print("=" * 100)
print("v9.4 FINAL SUMMARY — ALL EXPERIMENTS")
print("=" * 100)

# Add baselines for reference
# v9 original (from v9.3 comparison)
ALL_RESULTS.insert(0, {'experiment': 'v9_original_noisy',
    'macro_f1': 0.5806, 'N_f1': 0.933, 'S_f1': 0.171, 'S_recall': 0.135,
    'S_precision': 0.233, 'V_f1': 0.638, 'V_recall': 0.890, 'set': 'v9 baseline'})
# v9.3 (0.95 cleaning)
ALL_RESULTS.insert(1, {'experiment': 'v93_thr0.95',
    **phase_A_results[0.95]['test_metrics']})

df = pd.DataFrame(ALL_RESULTS)
display_cols = ['experiment', 'macro_f1', 'S_f1', 'S_recall', 'S_precision', 'V_f1']
df_d = df[display_cols].copy()
for c in display_cols[1:]:
    df_d[c] = df_d[c].astype(float).round(4)
print(df_d.to_string(index=False))

print(f"\n{'='*100}")
print(f"VERDICT")
print(f"{'='*100}")

# Best S F1
best_s = df.loc[df['S_f1'].astype(float).idxmax()]
best_macro = df.loc[df['macro_f1'].astype(float).idxmax()]
print(f"\nBest S F1     : {best_s['experiment']} → {float(best_s['S_f1']):.4f}")
print(f"Best Macro F1 : {best_macro['experiment']} → {float(best_macro['macro_f1']):.4f}")
print(f"v9 baseline   : S F1 = 0.1712, Macro F1 = 0.5806")

print(f"\n{'─'*100}")
print(f"{'Experiment':<35} {'S F1':>8} {'Δ from v9':>10} {'S recall':>10} {'S prec':>8}")
print(f"{'─'*75}")
for _, row in df.iterrows():
    d_s = float(row['S_f1']) - 0.1712
    marker = '★' if d_s > 0.03 else ('✓' if d_s > 0 else ' ')
    print(f"  {marker} {row['experiment']:<31} {float(row['S_f1']):>8.4f} {d_s:>+10.4f} "
          f"{float(row['S_recall']):>10.4f} {float(row['S_precision']):>8.4f}")

# Save
final = {
    'timestamp': datetime.now().isoformat(),
    'best_cleaning_threshold': BEST_THRESHOLD,
    'phase_A_threshold_sweep': {str(t): {k:v for k,v in r.items() if k != 'model'}
                                  for t, r in phase_A_results.items()},
    'phase_B_fold_stability': {str(k): v for k, v in fold_results.items()},
    'phase_C_window200': test_200,
    'all_results': [{k:v for k,v in r.items()} for r in ALL_RESULTS],
}
with open(f'{OUTPUTS_V94}/v94_summary.json', 'w') as f:
    json.dump(final, f, indent=2, default=str)
print(f"\nSaved: {OUTPUTS_V94}/v94_summary.json")


v9.4 FINAL SUMMARY — ALL EXPERIMENTS
         experiment  macro_f1   S_f1  S_recall  S_precision   V_f1
  v9_original_noisy    0.5806 0.1710    0.1350       0.2330 0.6380
        v93_thr0.95    0.5592 0.1991    0.1572       0.2713 0.5666
  v9_original_noisy    0.5806 0.1710    0.1350       0.2330 0.6380
        v93_thr0.95    0.5592 0.1991    0.1572       0.2713 0.5666
          A_thr0.85    0.5518 0.1780    0.1619       0.1978 0.5647
          A_thr0.95    0.5592 0.1991    0.1572       0.2713 0.5666
          A_thr1.05    0.5358 0.1311    0.0982       0.1969 0.5678
C_window200_thr0.95    0.5829 0.0000    0.0000       0.0000 0.7771

VERDICT

Best S F1     : v93_thr0.95 → 0.1991
Best Macro F1 : C_window200_thr0.95 → 0.5829
v9 baseline   : S F1 = 0.1712, Macro F1 = 0.5806

────────────────────────────────────────────────────────────────────────────────────────────────────
Experiment                              S F1  Δ from v9   S recall   S prec
─────────────────────────────────────────

## What to look at

### Phase A (threshold sweep)
- If threshold 1.05 gives higher S F1 than 0.95, the aggressive cleaning was removing useful training signal. Use 1.05 going forward.
- If 0.85 gives higher S precision but lower recall, you're trading coverage for accuracy — pick based on clinical priority.
- If all three are within ±0.02, the cleaning threshold doesn't matter much — the bottleneck is elsewhere.

### Phase B (3-fold stability)
- If the 3 folds pick similar thresholds (spread < 0.20), the 4-patient val was the problem. A bigger val set fixes the threshold selection.
- If the folds disagree (spread > 0.20), the model's optimal operating point is patient-dependent. This means the architecture can't generalize — no threshold will work well across all patients.
- The key comparison: 3-fold mean S recall vs test S recall. If 3-fold is still 2-3× higher than test, the model overfits to train-patient morphology regardless of val size.

### Phase C (window 200)
- If S F1 @ 200 > S F1 @ 130 by more than 0.03, the longer window helps. The KB was right — window size matters.
- If S F1 @ 200 ≈ S F1 @ 130, the window isn't the bottleneck. The model can't use the extra context.
- If S F1 @ 200 < S F1 @ 130, the longer window hurts (more noise, harder to learn). Stick with 130.

### The honest expectation
v9.3 showed the val/test gap is 4.3× on S recall. That's a generalization problem, and none of these three experiments directly adds more patient diversity to training. They test whether the generalization gap is fixable by:
- Better label quality (Phase A — already mostly done in v9.3)
- Better validation (Phase B — diagnostic only)
- More input signal (Phase C — the KB's hypothesis)

If none of these close the gap, the next step is either:
1. **More training patients** — add PTB-XL or CPSC2018 (different distributions, more S diversity)
2. **Different architecture** — the conv trunk may be fundamentally limited for cross-patient S generalization
3. **Accept the limitation** — report v9.3 as the best achievable on MIT-BIH alone, with the caveat that S recall is patient-dependent
